In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from tdc.benchmark_group import admet_group

In [2]:
BENCHMARK_NAME = "Solubility_AqSolDB"

PROCESSED_DIR = Path("../data/processed/aqsoldb")

RANDOM_STATE = 42

In [5]:
feature_sets = {
    "descriptorsRO5": pd.read_parquet(
        PROCESSED_DIR / "train_val_descriptors_RO5.parquet"
    ),

    "descriptorsESOL": pd.read_parquet(
        PROCESSED_DIR / "train_val_descriptors_ESOL.parquet"
    ),

    "descriptorsEXT": pd.read_parquet(
        PROCESSED_DIR / "train_val_descriptors_extended.parquet"
    ),

    "maccs": pd.read_parquet(
        PROCESSED_DIR / "train_val_maccs.parquet"
    ),

    "morgan": pd.read_parquet(
        PROCESSED_DIR / "train_val_morgan.parquet"
    ),

    "descRO5_maccs": pd.read_parquet(
        PROCESSED_DIR / "train_val_descRO5_maccs.parquet"
    ),

    "descESOL_maccs": pd.read_parquet(
        PROCESSED_DIR / "train_val_descESOL_maccs.parquet"
    ),

    "descEXT_maccs": pd.read_parquet(
        PROCESSED_DIR / "train_val_descEXT_maccs.parquet"
    ),

    "descRO5_morgan": pd.read_parquet(
        PROCESSED_DIR / "train_val_descRO5_morgan.parquet"
    ),

    "descESOL_morgan": pd.read_parquet(
        PROCESSED_DIR / "train_val_descESOL_morgan.parquet"
    ),

    "descEXT_morgan": pd.read_parquet(
        PROCESSED_DIR / "train_val_descEXT_morgan.parquet"
    ),
}

In [6]:
y = pd.read_parquet(
    PROCESSED_DIR / "train_val_target.parquet"
)["Y"]

In [7]:
metadata = pd.read_csv(
    PROCESSED_DIR / "train_val_metadata.csv"
)

In [8]:
group = admet_group(path="../data/raw/")

train_split, valid_split = group.get_train_valid_split(
    benchmark=BENCHMARK_NAME,
    split_type="default",
    seed=RANDOM_STATE,
)

Found local copy...
generating training, validation splits...
100%|██████████| 7985/7985 [00:01<00:00, 4907.44it/s]


In [9]:
train_drugs = set(train_split["Drug"])
valid_drugs = set(valid_split["Drug"])

train_mask = metadata["Drug"].isin(train_drugs)
valid_mask = metadata["Drug"].isin(valid_drugs)

In [10]:
print("Train:", train_mask.sum())
print("Validation:", valid_mask.sum())

assert (train_mask & valid_mask).sum() == 0
assert train_mask.sum() + valid_mask.sum() == len(metadata)

Train: 5045
Validation: 2940


In [11]:
y_train = y.loc[train_mask].reset_index(drop=True)
y_valid = y.loc[valid_mask].reset_index(drop=True)

In [14]:
def evaluate_regression(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(
            y_true,
            y_pred,
        ),

        "RMSE": mean_squared_error(
            y_true,
            y_pred,
        ) ** 0.5,

        "R2": r2_score(
            y_true,
            y_pred,
        ),
    }

In [15]:
models = {
    "dummy": DummyRegressor(
        strategy="mean"
    ),

    "ridge": Pipeline([
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "model",
            Ridge(alpha=1.0),
        ),
    ]),

    "random_forest": RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),

    "gradient_boosting": GradientBoostingRegressor(
        random_state=RANDOM_STATE,
    ),
}

In [25]:
results = []

for feature_name, X in feature_sets.items():

    X_train = X.loc[
        train_mask
    ].reset_index(drop=True)

    X_valid = X.loc[
        valid_mask
    ].reset_index(drop=True)

    for model_name, model in models.items():

        model.fit(
            X_train,
            y_train,
        )

        y_pred = model.predict(
            X_valid
        )

        metrics = evaluate_regression(
            y_valid,
            y_pred,
        )

        results.append({
            "features": feature_name,
            "model": model_name,
            **metrics,
        })

In [26]:
results_df = pd.DataFrame(results)

results_df.sort_values("MAE")

,features,model,MAE,RMSE,R2
22,descRO5_maccs,random_forest,0.818394,1.115924,0.730444
30,descEXT_maccs,random_forest,0.824893,1.132947,0.722157
26,descESOL_maccs,random_forest,0.831206,1.139605,0.718882
31,descEXT_maccs,gradient_boosting,0.843081,1.141035,0.718176
27,descESOL_maccs,gradient_boosting,0.843874,1.138822,0.719268
23,descRO5_maccs,gradient_boosting,0.846216,1.140892,0.718247
42,descEXT_morgan,random_forest,0.855579,1.182280,0.697434
34,descRO5_morgan,random_forest,0.856271,1.177118,0.700070
38,descESOL_morgan,random_forest,0.866732,1.202099,0.687205
39,descESOL_morgan,gradient_boosting,0.886739,1.195314,0.690726


In [27]:
FEATURES_TO_EVALUATE = [
    "descRO5_maccs",
    "descESOL_maccs",
    "descEXT_maccs",
]

SEEDS = [0, 1, 2, 3, 4]

MULTI_SEED_MODELS = {
    "gradient_boosting": GradientBoostingRegressor(
        random_state=RANDOM_STATE,
    ),

    "random_forest": RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

multi_seed_results = []

for seed in SEEDS:

    # Generate the TDC scaffold split for this seed
    train_split, valid_split = group.get_train_valid_split(
        benchmark=BENCHMARK_NAME,
        split_type="default",
        seed=seed,
    )

    train_drugs = set(train_split["Drug"])
    valid_drugs = set(valid_split["Drug"])

    train_mask = metadata["Drug"].isin(train_drugs)
    valid_mask = metadata["Drug"].isin(valid_drugs)

    assert not (train_mask & valid_mask).any()
    assert train_mask.sum() + valid_mask.sum() == len(metadata)

    y_train = (
        y.loc[train_mask]
        .reset_index(drop=True)
    )

    y_valid = (
        y.loc[valid_mask]
        .reset_index(drop=True)
    )

    for feature_name in FEATURES_TO_EVALUATE:

        X = feature_sets[feature_name]

        X_train = (
            X.loc[train_mask]
            .reset_index(drop=True)
        )

        X_valid = (
            X.loc[valid_mask]
            .reset_index(drop=True)
        )

        for model_name, model in MULTI_SEED_MODELS.items():

            model.fit(
                X_train,
                y_train,
            )

            y_pred = model.predict(X_valid)

            metrics = evaluate_regression(
                y_valid,
                y_pred,
            )

            multi_seed_results.append({
                "seed": seed,
                "features": feature_name,
                "model": model_name,
                **metrics,
            })

generating training, validation splits...
100%|██████████| 7985/7985 [00:01<00:00, 5493.12it/s] 
generating training, validation splits...
100%|██████████| 7985/7985 [00:01<00:00, 5151.12it/s]
generating training, validation splits...
100%|██████████| 7985/7985 [00:01<00:00, 5336.64it/s] 
generating training, validation splits...
100%|██████████| 7985/7985 [00:01<00:00, 4419.22it/s]
generating training, validation splits...
100%|██████████| 7985/7985 [00:01<00:00, 5105.28it/s]


In [28]:
seed_results_df = pd.DataFrame(
    multi_seed_results
)

seed_results_df.sort_values(
    ["seed", "MAE"]
)

,seed,features,model,MAE,RMSE,R2
0,0,descRO5_maccs,gradient_boosting,0.647593,0.874249,0.677114
2,0,descESOL_maccs,gradient_boosting,0.651200,0.882857,0.670724
5,0,descEXT_maccs,random_forest,0.653421,0.883407,0.670314
1,0,descRO5_maccs,random_forest,0.671663,0.900791,0.657210
3,0,descESOL_maccs,random_forest,0.672087,0.900688,0.657289
4,0,descEXT_maccs,gradient_boosting,0.680168,0.909312,0.650694
11,1,descEXT_maccs,random_forest,0.884480,1.233975,0.741682
9,1,descESOL_maccs,random_forest,0.885304,1.233393,0.741926
7,1,descRO5_maccs,random_forest,0.892515,1.248663,0.735496
10,1,descEXT_maccs,gradient_boosting,0.922548,1.260299,0.730544


In [29]:
summary_df = (
    seed_results_df
    .groupby(
        ["features", "model"]
    )
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        R2_mean=("R2", "mean"),
        R2_std=("R2", "std"),
    )
    .sort_values("MAE_mean")
)

summary_df

MAE_mean   MAE_std  RMSE_mean  RMSE_std  \
features       model                                                        
descEXT_maccs  random_forest      0.797280  0.085870   1.095711  0.130385   
descESOL_maccs random_forest      0.803988  0.079348   1.104358  0.123269   
descRO5_maccs  random_forest      0.807260  0.081538   1.106138  0.127363   
               gradient_boosting  0.826464  0.105474   1.122933  0.149788   
descESOL_maccs gradient_boosting  0.831952  0.105809   1.128489  0.147408   
descEXT_maccs  gradient_boosting  0.838014  0.092764   1.133802  0.134048   

                                   R2_mean    R2_std  
features       model                                  
descEXT_maccs  random_forest      0.745223  0.059986  
descESOL_maccs random_forest      0.740835  0.062131  
descRO5_maccs  random_forest      0.740496  0.060589  
               gradient_boosting  0.734933  0.053284  
descESOL_maccs gradient_boosting  0.732305  0.053363  
descEXT_maccs  gradient_boosting  0.728609  0.057900